# 10 -- Baseline vs. candidate release comparison

Paired script: `analysis/compare_releases.py`. Two-sample bootstrap confidence interval on
the difference in win rate / R-expectancy between a baseline and candidate release. Per the
reproducibility contract's "tiny samples cannot drive automatic changes" rule, this never
declares a release "better" automatically -- it reports a difference and its CI; the go/
no-go judgment remains a human decision.

**Uses clearly-labelled SYNTHETIC trade data for both releases.** Real-data run: PENDING.

In [ ]:
import sys
import tempfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.compare_releases import run

In [ ]:
def make_trades(path, exits, profits):
    rows = [
        {"trade_id": f"t{i}", "symbol": "XAUUSD", "is_long": "True",
         "entry_time": "2026-07-21T00:00:00Z", "exit_time": "2026-07-21T01:00:00Z",
         "entry_price": 100.0, "exit_price": e, "stop_price": 98.0, "profit": p}
        for i, (e, p) in enumerate(zip(exits, profits))
    ]
    pd.DataFrame(rows).to_csv(path, index=False)

tmp_dir = Path(tempfile.mkdtemp(prefix="themba_compare_demo_"))
# Baseline: 25% win rate (5 wins / 15 losses out of 20). Candidate: 75% (15/20).
make_trades(tmp_dir / "baseline.csv", [95.0] * 15 + [105.0] * 5, [-10.0] * 15 + [10.0] * 5)
make_trades(tmp_dir / "candidate.csv", [105.0] * 15 + [95.0] * 5, [10.0] * 15 + [-10.0] * 5)

In [ ]:
summary = run(tmp_dir / "baseline.csv", tmp_dir / "candidate.csv", n_resamples=2000, seed=1,
              output_json=tmp_dir / "compare.json", repo_path=PROJECT_ROOT.parents[1])

print(f"baseline_win_rate    = {summary['baseline_win_rate']:.4f}")
print(f"candidate_win_rate   = {summary['candidate_win_rate']:.4f}")
print(f"win_rate_diff        = {summary['win_rate_diff']['observed_diff']:.4f} "
      f"(95% CI [{summary['win_rate_diff']['ci_lower']:.4f}, {summary['win_rate_diff']['ci_upper']:.4f}])")
print(f"likely_significant   = {summary['win_rate_diff']['likely_significant']}")

assert abs(summary["baseline_win_rate"] - 0.25) < 1e-9
assert abs(summary["candidate_win_rate"] - 0.75) < 1e-9
assert summary["win_rate_diff"]["likely_significant"] is True

## Real-data run: PENDING

Requires two real trade histories (baseline release vs. a candidate release) -- neither exists yet.